# Data Cleaning

Imports & configuration

In [21]:
import pandas as pd
import numpy as np
import os
import glob
import re

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)


Define raw and cleaned directories

In [22]:
RAW_PATH = Path(r"C:\Users\shukl\Documents\DS project\DataScienceProject\data\raw")
CLEAN_PATH = Path(r"C:\Users\shukl\Documents\DS project\DataScienceProject\data\processed")
CLEAN_PATH.mkdir(parents=True, exist_ok=True)

print("Raw path:", RAW_PATH)
print("Clean path:", CLEAN_PATH)

Raw path: C:\Users\shukl\Documents\DS project\DataScienceProject\data\raw
Clean path: C:\Users\shukl\Documents\DS project\DataScienceProject\data\processed


Find all files

In [23]:
files = list(RAW_PATH.glob("*"))

for file in files:
    print(file.name)

churn_labels.csv
content.csv
customers (3).csv
customer_feedback (2).csv
payments (2).csv
subscriptions (1).csv
subscription_plans.csv
support_tickets (1).csv
viewing_activity.csv


In [24]:
csv_files = sorted(RAW_PATH.glob("*.csv"))

print(f"Total CSV files found: {len(csv_files)}")

for i, file in enumerate(csv_files, start=1):
    print(f"{i}. {file.name}")

Total CSV files found: 9
1. churn_labels.csv
2. content.csv
3. customer_feedback (2).csv
4. customers (3).csv
5. payments (2).csv
6. subscription_plans.csv
7. subscriptions (1).csv
8. support_tickets (1).csv
9. viewing_activity.csv


2. Load all 9 datasets

In [25]:
datasets = {}

for file in csv_files:
    try:
        datasets[file.stem] = pd.read_csv(file)
        print(f"{file.name:<40} {datasets[file.stem].shape}")
        
    except Exception as e:
        print(f"ERROR reading {file.name}: {e}")

churn_labels.csv                         (8000, 4)
content.csv                              (500, 9)
customer_feedback (2).csv                (5046, 6)
customers (3).csv                        (8048, 12)
payments (2).csv                         (110735, 8)
subscription_plans.csv                   (3, 6)
subscriptions (1).csv                    (8875, 10)
support_tickets (1).csv                  (6120, 11)
viewing_activity.csv                     (66422, 9)


In [26]:
datasets["churn_labels"]

,customer_id,churned,churn_date,churn_reason
0,CUST000001,False,NaN,NaN
1,CUST000002,False,NaN,NaN
2,CUST000003,True,2026-04-20,Too expensive
3,CUST000004,False,NaN,NaN
4,CUST000005,False,NaN,NaN
...,...,...,...,...
7995,CUST007996,False,NaN,NaN
7996,CUST007997,False,NaN,NaN
7997,CUST007998,False,NaN,NaN
7998,CUST007999,False,NaN,NaN


3. Initial data profiling

In [27]:
profile = []

for name, df in datasets.items():

    profile.append({
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "missing_values": df.isnull().sum().sum(),
        "duplicate_rows": df.duplicated().sum(),
        "memory_mb": round(
            df.memory_usage(deep=True).sum() / 1024**2,
            2
        )
    })

profile_df = pd.DataFrame(profile)

profile_df

,dataset,rows,columns,missing_values,duplicate_rows,memory_mb
0,churn_labels,8000,4,12018,0,1.07
1,content,500,9,0,0,0.20
2,customer_feedback (2),5046,6,0,0,1.82
3,customers (3),8048,12,9517,48,4.47
4,payments (2),110735,8,214264,0,36.15
5,subscription_plans,3,6,0,0,0.00
6,subscriptions (1),8875,10,21768,0,3.47
7,support_tickets (1),6120,11,3056,0,3.71
8,viewing_activity,66422,9,1195,0,23.64


. Inspect columns and data types

In [28]:
for name, df in datasets.items():

    print("\n" + "=" * 80)
    print(f"DATASET: {name}")
    print("=" * 80)

    display(
        pd.DataFrame({
            "column": df.columns,
            "dtype": df.dtypes.astype(str),
            "missing": df.isnull().sum(),
            "missing_%": (
                df.isnull().mean() * 100
            ).round(2),
            "unique": df.nunique()
        })
    )


DATASET: churn_labels


,column,dtype,missing,missing_%,unique
customer_id,customer_id,str,0,0.00,8000
churned,churned,bool,0,0.00,2
churn_date,churn_date,str,6009,75.11,271
churn_reason,churn_reason,str,6009,75.11,11



DATASET: content


,column,dtype,missing,missing_%,unique
content_id,content_id,str,0,0.0,500
title,title,str,0,0.0,500
content_type,content_type,str,0,0.0,5
genre,genre,str,0,0.0,16
release_year,release_year,int64,0,0.0,29
duration_minutes,duration_minutes,int64,0,0.0,117
content_language,content_language,str,0,0.0,13
production_country,production_country,str,0,0.0,20
maturity_rating,maturity_rating,str,0,0.0,7



DATASET: customer_feedback (2)


...

,column,dtype,missing,missing_%,unique
subscription_id,subscription_id,str,0,0.00,8875
customer_id,customer_id,str,0,0.00,8000
plan_id,plan_id,str,0,0.00,3
subscription_start_date,subscription_start_date,str,0,0.00,2620
subscription_end_date,subscription_end_date,str,8000,90.14,670
subscription_status,subscription_status,str,0,0.00,3
auto_renew,auto_renew,bool,0,0.00,2
cancellation_date,cancellation_date,str,6884,77.57,271
cancellation_reason,cancellation_reason,str,6884,77.57,11
monthly_price,monthly_price,float64,0,0.00,3



DATASET: support_tickets (1)


,column,dtype,missing,missing_%,unique
ticket_id,ticket_id,str,0,0.00,6120
customer_id,customer_id,str,0,0.00,4408
ticket_date,ticket_date,str,0,0.00,1888
issue_category,issue_category,str,0,0.00,6
issue_subcategory,issue_subcategory,str,0,0.00,24
priority,priority,str,0,0.00,4
ticket_status,ticket_status,str,0,0.00,4
resolution_time_hours,resolution_time_hours,float64,1528,24.97,667
customer_satisfaction_score,customer_satisfaction_score,float64,1528,24.97,5
ticket_description,ticket_description,str,0,0.00,32



DATASET: viewing_activity


,column,dtype,missing,missing_%,unique
viewing_id,viewing_id,str,0,0.0,66422
customer_id,customer_id,str,0,0.0,7980
content_id,content_id,str,0,0.0,500
viewing_date,viewing_date,str,0,0.0,365
watch_duration_minutes,watch_duration_minutes,int64,0,0.0,273
completion_percentage,completion_percentage,float64,531,0.8,100
device_type,device_type,str,664,1.0,23
login_location,login_location,str,0,0.0,83
session_duration_minutes,session_duration_minutes,int64,0,0.0,285


5. Create reusable column-name cleaning function

In [29]:
def clean_column_names(df):

    df = df.copy()

    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r"[^a-zA-Z0-9]+", "_", regex=True)
        .str.strip("_")
    )

    return df

8. Clean string values

In [32]:
def clean_string_columns(df):

    df = df.copy()

    string_columns = df.select_dtypes(
        include=["object", "string"]
    ).columns

    for col in string_columns:

        df[col] = df[col].apply(
            lambda x: x.strip()
            if isinstance(x, str)
            else x
        )

    return df

Remove completely empty rows and columns, while preserving empty identifier (`*_id`) columns so later reconciliation steps can populate them.

In [38]:
def remove_empty_rows_columns(df):

    df = df.copy()

    df = df.dropna(how="all")

    empty_columns = df.columns[df.isna().all()]
    columns_to_preserve = [
        column for column in empty_columns
        if column.endswith("_id")
    ]
    df = df.dropna(axis=1, how="all")
    for column in columns_to_preserve:
        df[column] = pd.Series(pd.NA, index=df.index, dtype="string")

    return df

Remove duplicate rows

In [31]:
def remove_duplicate_rows(df):

    df = df.copy()

    duplicate_count = df.duplicated().sum()

    print("Duplicate rows found:", duplicate_count)

    df = df.drop_duplicates()

    return df

Convert empty strings to NaN

In [33]:
def convert_empty_to_nan(df):

    df = df.copy()

    df = df.replace(
        r"^\s*$",
        np.nan,
        regex=True
    )

    return df

Handle missing values

In [ ]:
def handle_missing_values(df):

    df = df.copy()

    # Numeric columns
    numeric_columns = df.select_dtypes(
        include=np.number
    ).columns

    for col in numeric_columns:

        if df[col].isnull().any():

            median_value = df[col].median()

            df[col] = df[col].fillna(
                median_value
            )

    # Categorical / string columns
    categorical_columns = df.select_dtypes(
        include=["object", "string", "category"]
    ).columns

    for col in categorical_columns:

        if df[col].isnull().any():

            mode = df[col].mode()

            if not mode.empty:
                df[col] = df[col].fillna(
                    mode.iloc[0]
                )
            else:
                df[col] = df[col].fillna(
                    "Unknown"
                )

    return df

## 9. Data Cleaning & Feature Engineering

We will work through the checklist in small, ordered steps. Each section explains the decision, why it is useful, and how to defend it in an interview. Shared formatting cleanup is applied to all nine datasets; the specific rules below are applied only to datasets that contain the relevant fields.

### 9.1 Prepare working datasets

The next code cell builds cleaned copies of the loaded datasets and checks that the required columns exist. Keeping separate copies protects the original `datasets` and makes it easier to compare raw and cleaned data.

**Interview reasoning:** Validate schemas early. A clear error at the start is easier to debug than a confusing failure deep inside a transformation.

In [ ]:
# Apply shared cleanup to every loaded dataset and keep the raw frames unchanged.
cleaned_datasets = {
    name: remove_empty_rows_columns(
        convert_empty_to_nan(
            clean_string_columns(clean_column_names(frame))
        )
    )
    for name, frame in datasets.items()
}


def dataset_name(prefix):
    matches = [
        name for name in cleaned_datasets
        if name.lower().startswith(prefix.lower())
    ]
    if len(matches) != 1:
        raise ValueError(
            f"Expected one dataset starting with {prefix!r}; found {matches}"
        )
    return matches[0]


def require_columns(frame, columns, dataset_label):
    missing = [column for column in columns if column not in frame.columns]
    if missing:
        raise KeyError(f"{dataset_label} is missing required columns: {missing}")


customer_key = dataset_name("customers")
viewing_key = dataset_name("viewing_activity")
tickets_key = dataset_name("support_tickets")
subscriptions_key = dataset_name("subscriptions")
payments_key = dataset_name("payments")

customers_clean = cleaned_datasets[customer_key].copy()
viewing_clean = cleaned_datasets[viewing_key].copy()
tickets_clean = cleaned_datasets[tickets_key].copy()
subscriptions_clean = cleaned_datasets[subscriptions_key].copy()
payments_clean = cleaned_datasets[payments_key].copy()

required_columns = {
    customer_key: [
        "customer_id", "age", "country", "city",
        "preferred_language", "registration_date",
    ],
    viewing_key: [
        "customer_id", "watch_duration_minutes",
        "completion_percentage", "device_type",
        "session_duration_minutes", "viewing_id",
    ],
    tickets_key: ["ticket_date", "customer_satisfaction_score"],
    subscriptions_key: [
        "subscription_id", "customer_id",
        "subscription_start_date", "subscription_end_date",
    ],
    payments_key: [
        "payment_id", "customer_id", "subscription_id",
        "payment_date", "amount",
    ],
}

for dataset_label, columns in required_columns.items():
    require_columns(cleaned_datasets[dataset_label], columns, dataset_label)

print(f"Prepared {len(cleaned_datasets)} datasets.")
print("Checklist-specific datasets:", customer_key, viewing_key, tickets_key, subscriptions_key, payments_key)

Prepared 9 datasets.
Checklist-specific datasets: customers (3) viewing_activity support_tickets (1) subscriptions (1) payments (2)


### 9.2 Standardize category labels

Country names and device types can have multiple spellings for the same category. We normalize punctuation and whitespace before applying a small alias dictionary. Unrecognized values are title-cased, while genuinely missing values remain missing for the imputation step.

**Strategy:** Use explicit aliases for known equivalents instead of fuzzy matching. This avoids silently merging values that only look similar.

**Interview answer:** “I normalized known category variants with a controlled mapping, inspected the resulting categories, and kept unknown values separate rather than guessing.”

In [42]:
country_aliases = {
    "uk": "United Kingdom",
    "u k": "United Kingdom",
    "great britain": "United Kingdom",
    "britain": "United Kingdom",
    "usa": "United States",
    "u s": "United States",
    "u s a": "United States",
    "united states of america": "United States",
}

device_aliases = {
    "smart tv": "Smart TV",
    "smarttv": "Smart TV",
    "tv": "Smart TV",
    "television": "Smart TV",
    "mobile phone": "Mobile",
    "smartphone": "Mobile",
    "cell phone": "Mobile",
    "phone": "Mobile",
    "desktop computer": "Desktop",
    "pc": "Desktop",
    "notebook": "Laptop",
    "ipad": "Tablet",
    "console": "Gaming Console",
    "playstation xbox": "Gaming Console",
}


def normalize_category(value, aliases):
    if pd.isna(value):
        return np.nan

    cleaned = " ".join(str(value).casefold().replace(".", "").split())
    lookup_key = re.sub(r"[^a-z0-9]+", " ", cleaned).strip()
    return aliases.get(lookup_key, cleaned.title())


customers_clean["country"] = customers_clean["country"].map(
    lambda value: normalize_category(value, country_aliases)
)
viewing_clean["device_type"] = viewing_clean["device_type"].map(
    lambda value: normalize_category(value, device_aliases)
)

print("Canonical country values:")
print(customers_clean["country"].value_counts(dropna=False).head(15).to_string())
print("\nCanonical device values:")
print(viewing_clean["device_type"].value_counts(dropna=False).to_string())

Canonical country values:
country
United States     1404
India             1164
United Kingdom     617
NaN                409
Canada             404
Brazil             370
Germany            360
France             298
Mexico             282
Japan              260
Philippines        232
Indonesia          231
Italy              220
Nigeria            205
South Africa       204

Canonical device values:
device_type
Smart TV          25132
Mobile            19822
Laptop             8333
Tablet             7811
Gaming Console     3959
Desktop             701
NaN                 664


### 9.3 Detect and resolve duplicate customers

`customer_id` is the customer key, so repeated IDs represent duplicate records for this task. When duplicates exist, keep the row with the most populated fields; if completeness ties, prefer the latest valid `registration_date`. Missing IDs are not useful as entity keys and are excluded.

**Why this strategy:** Dropping every repeated row arbitrarily could keep an incomplete record. Sorting by completeness makes the rule explicit and repeatable.

**Interview answer:** “I defined the business key first, measured duplicate IDs, then used completeness and a date tie-breaker to retain one record per customer. In production I would also investigate why duplicates were created.”

In [43]:
customer_rows_before_dedup = len(customers_clean)
customer_duplicate_rows = int(
    customers_clean["customer_id"].duplicated(keep=False).sum()
)
customer_duplicate_ids = int(
    customers_clean.loc[
        customers_clean["customer_id"].duplicated(keep=False), "customer_id"
    ].nunique()
)

customers_clean["registration_date"] = pd.to_datetime(
    customers_clean["registration_date"], errors="coerce"
)
customers_clean["_completeness"] = customers_clean.notna().sum(axis=1)
customers_clean = (
    customers_clean.sort_values(
        ["_completeness", "registration_date"],
        ascending=[False, False],
        na_position="last",
        kind="mergesort",
    )
    .dropna(subset=["customer_id"])
    .drop_duplicates(subset="customer_id", keep="first")
    .drop(columns="_completeness")
    .reset_index(drop=True)
)
customer_duplicate_rows_removed = customer_rows_before_dedup - len(customers_clean)

print(f"Rows participating in duplicate IDs: {customer_duplicate_rows}")
print(f"Duplicate customer IDs: {customer_duplicate_ids}")
print(f"Rows removed during deduplication: {customer_duplicate_rows_removed}")
print(f"Unique customer IDs after cleanup: {customers_clean['customer_id'].nunique()}")

Rows participating in duplicate IDs: 96
Duplicate customer IDs: 48
Rows removed during deduplication: 48
Unique customer IDs after cleanup: 8000


### 9.4 Handle missing values

| Dataset field | Strategy | Reason |
|---|---|---|
| `customers.age` | Median | Robust to skew and extreme ages. |
| `customers.country`, `city`, `preferred_language` | `Unknown` | Keeps the row without inventing a category. |
| `viewing_activity.completion_percentage` | Median | Uses a typical observed completion value. |
| `viewing_activity.device_type` | `Unknown` | Preserves missingness as an explicit category. |
| `support_tickets.customer_satisfaction_score` | Median | Does not assume a positive or negative rating. |

The code reports missing counts **before** filling them. If a numeric column is entirely missing, its median is undefined, so this cell leaves those values missing rather than fabricating a number.

**Interview answer:** “I chose robust numeric imputations and explicit categorical placeholders, but I would fit imputation values on the training split only to avoid data leakage in a predictive pipeline.”

In [44]:
imputation_counts = {}

customers_clean["age"] = pd.to_numeric(
    customers_clean["age"], errors="coerce"
)
imputation_counts["customers.age"] = int(customers_clean["age"].isna().sum())
age_median = customers_clean["age"].median()
if pd.notna(age_median):
    customers_clean["age"] = customers_clean["age"].fillna(age_median)

for column in ["country", "city", "preferred_language"]:
    imputation_counts[f"customers.{column}"] = int(
        customers_clean[column].isna().sum()
    )
    customers_clean[column] = customers_clean[column].fillna("Unknown")

viewing_clean["completion_percentage"] = pd.to_numeric(
    viewing_clean["completion_percentage"], errors="coerce"
)
imputation_counts["viewing_activity.completion_percentage"] = int(
    viewing_clean["completion_percentage"].isna().sum()
)
completion_median = viewing_clean["completion_percentage"].median()
if pd.notna(completion_median):
    viewing_clean["completion_percentage"] = viewing_clean[
        "completion_percentage"
    ].fillna(completion_median)

imputation_counts["viewing_activity.device_type"] = int(
    viewing_clean["device_type"].isna().sum()
)
viewing_clean["device_type"] = viewing_clean["device_type"].fillna("Unknown")

tickets_clean["customer_satisfaction_score"] = pd.to_numeric(
    tickets_clean["customer_satisfaction_score"], errors="coerce"
)
imputation_counts["support_tickets.customer_satisfaction_score"] = int(
    tickets_clean["customer_satisfaction_score"].isna().sum()
)
satisfaction_median = tickets_clean["customer_satisfaction_score"].median()
if pd.notna(satisfaction_median):
    tickets_clean["customer_satisfaction_score"] = tickets_clean[
        "customer_satisfaction_score"
    ].fillna(satisfaction_median)

print("Missing-value counts before imputation:")
print(pd.Series(imputation_counts, name="missing_count").to_string())

Missing-value counts before imputation:
customers.age                                   321
customers.country                               409
customers.city                                  483
customers.preferred_language                    251
viewing_activity.completion_percentage          531
viewing_activity.device_type                    664
support_tickets.customer_satisfaction_score    1528


### 9.5 Validate support-ticket dates

Parse `ticket_date` using pandas date parsing with invalid values coerced to `NaT` (pandas' missing-date value). Because a malformed ticket date cannot be reliably inferred from the available fields, remove those rows and report the count.

**Interview reasoning:** Do not silently replace invalid dates with today's date or another arbitrary value; that would create false chronology and distort time-based analysis.

In [45]:
ticket_rows_before_date_check = len(tickets_clean)
tickets_clean["ticket_date"] = pd.to_datetime(
    tickets_clean["ticket_date"], errors="coerce"
)
invalid_ticket_date_count = int(tickets_clean["ticket_date"].isna().sum())
tickets_clean = tickets_clean.dropna(subset=["ticket_date"]).reset_index(drop=True)

print(f"Ticket rows before date validation: {ticket_rows_before_date_check}")
print(f"Rows removed for invalid ticket_date: {invalid_ticket_date_count}")
print(f"Ticket rows retained: {len(tickets_clean)}")

Ticket rows before date validation: 6120
Rows removed for invalid ticket_date: 61
Ticket rows retained: 6059


### 9.6 Detect and cap outliers

Use the interquartile range (IQR) rule: values below $Q_1 - 1.5 \times IQR$ or above $Q_3 + 1.5 \times IQR$ are flagged. Cap flagged values at the calculated fences (winsorization) rather than deleting their records. Since age, duration, and payment amount cannot be negative, the lower cap is at least zero.

**Why cap instead of drop?** A large value may be a valid long session or a high-priced payment. Capping limits its influence while preserving the row.

**Interview answer:** “I used a transparent univariate IQR screen, reviewed the counts and thresholds, and winsorized instead of dropping records. I would validate flagged cases with domain owners before production use.”

In [46]:
def cap_iqr_outliers(frame, column):
    values = pd.to_numeric(frame[column], errors="coerce")
    first_quartile = values.quantile(0.25)
    third_quartile = values.quantile(0.75)

    if pd.isna(first_quartile) or pd.isna(third_quartile):
        return {
            "column": column,
            "outlier_count": 0,
            "lower_cap": np.nan,
            "upper_cap": np.nan,
        }

    iqr = third_quartile - first_quartile
    lower_cap = max(0, first_quartile - 1.5 * iqr)
    upper_cap = third_quartile + 1.5 * iqr
    outlier_mask = values.notna() & (
        (values < lower_cap) | (values > upper_cap)
    )

    frame[column] = values.clip(lower=lower_cap, upper=upper_cap)
    return {
        "column": column,
        "outlier_count": int(outlier_mask.sum()),
        "lower_cap": round(float(lower_cap), 3),
        "upper_cap": round(float(upper_cap), 3),
    }


outlier_report = []
for frame, column in [
    (customers_clean, "age"),
    (viewing_clean, "watch_duration_minutes"),
    (viewing_clean, "session_duration_minutes"),
    (payments_clean, "amount"),
]:
    outlier_report.append(cap_iqr_outliers(frame, column))

print(pd.DataFrame(outlier_report).to_string(index=False))

                  column  outlier_count  lower_cap  upper_cap
                     age             43        2.5      62.50
  watch_duration_minutes            443        0.0     113.50
session_duration_minutes            357        0.0     125.50
                  amount            463        0.0      28.24


### 9.7 Reconcile payment subscription IDs

For each payment with a blank `subscription_id`, find subscriptions belonging to the same customer where `payment_date` falls inside the subscription's start/end dates. The endpoints are inclusive. If multiple subscription windows qualify, choose the one with the latest start date. Leave unmatched payments blank and count them; do not guess.

**Interview answer:** “I used both customer identity and temporal validity as join conditions. A customer-only join could assign a payment to the wrong renewal period. I also made overlap and unmatched-record behavior explicit.”

In [47]:
subscriptions_clean["subscription_start_date"] = pd.to_datetime(
    subscriptions_clean["subscription_start_date"], errors="coerce"
)
subscriptions_clean["subscription_end_date"] = pd.to_datetime(
    subscriptions_clean["subscription_end_date"], errors="coerce"
)
payments_clean["payment_date"] = pd.to_datetime(
    payments_clean["payment_date"], errors="coerce"
)

blank_subscription_mask = payments_clean["subscription_id"].isna()
payments_missing_subscription = payments_clean.loc[
    blank_subscription_mask, ["customer_id", "payment_date"]
].copy()
payments_missing_subscription["payment_row"] = payments_missing_subscription.index

subscription_candidates = payments_missing_subscription.merge(
    subscriptions_clean[
        [
            "subscription_id",
            "customer_id",
            "subscription_start_date",
            "subscription_end_date",
        ]
    ],
    on="customer_id",
    how="left",
)

inside_subscription_window = (
    subscription_candidates["subscription_id"].notna()
    & subscription_candidates["payment_date"].notna()
    & subscription_candidates["subscription_start_date"].notna()
    & (
        subscription_candidates["subscription_start_date"]
        <= subscription_candidates["payment_date"]
    )
    & (
        subscription_candidates["subscription_end_date"].isna()
        | (
            subscription_candidates["payment_date"]
            <= subscription_candidates["subscription_end_date"]
        )
    )
)

subscription_matches = (
    subscription_candidates.loc[inside_subscription_window]
    .sort_values(
        ["payment_row", "subscription_start_date"],
        ascending=[True, False],
        kind="mergesort",
    )
    .drop_duplicates(subset="payment_row", keep="first")
    .set_index("payment_row")["subscription_id"]
)
payments_clean.loc[subscription_matches.index, "subscription_id"] = (
    subscription_matches
)

matched_payment_count = len(subscription_matches)
unmatched_blank_subscription_count = int(
    payments_clean["subscription_id"].isna().sum()
)
print(f"Blank payment IDs matched: {matched_payment_count}")
print(f"Blank payment IDs still unmatched: {unmatched_blank_subscription_count}")

Blank payment IDs matched: 110735
Blank payment IDs still unmatched: 0


### 9.8 Build customer-level viewing features

Convert each viewing event into one row per customer with session count, total watch time, mean completion percentage, and distinct device count. Aggregation changes the grain from **one row per viewing session** to **one row per customer**, which is useful for customer-level analysis and modeling.

**Interview answer:** “I checked the unit of analysis before aggregating, grouped by customer ID, and chose features that summarize engagement. These features can be joined to customer data using the same key.”

In [49]:
viewing_clean["session_duration_minutes"] = pd.to_numeric(
    viewing_clean["session_duration_minutes"], errors="coerce"
)
customer_viewing_features = (
    viewing_clean.groupby("customer_id", dropna=False)
    .agg(
        session_count=("viewing_id", "count"),
        total_watch_duration_minutes=("watch_duration_minutes", "sum"),
        mean_completion_percentage=("completion_percentage", "mean"),
        distinct_device_count=("device_type", "nunique"),
    )
    .reset_index()
)

print("Customer-level viewing features:")
print(customer_viewing_features.head().to_string(index=False))

Customer-level viewing features:
customer_id  session_count  total_watch_duration_minutes  mean_completion_percentage  distinct_device_count
 CUST000001              5                         226.0                       77.20                      3
 CUST000002              5                         206.0                       65.60                      3
 CUST000003              2                          84.0                       76.50                      1
 CUST000004              8                         471.0                       63.75                      5
 CUST000005             10                         397.0                       70.00                      5


### 9.9 Checkpoint export for the first five datasets

This checkpoint saves the first five dataset-specific cleaning results alongside the shared-cleaned versions of the other files. Sections 9.10–9.14 apply their own rules to the remaining four datasets, and the final export at the end overwrites this checkpoint with the complete results.

In [50]:
cleaned_datasets[customer_key] = customers_clean
cleaned_datasets[viewing_key] = viewing_clean
cleaned_datasets[tickets_key] = tickets_clean
cleaned_datasets[subscriptions_key] = subscriptions_clean
cleaned_datasets[payments_key] = payments_clean

CLEAN_PATH.mkdir(parents=True, exist_ok=True)
for dataset_label, frame in cleaned_datasets.items():
    output_stem = re.sub(r"\s*\(\d+\)$", "", dataset_label).replace(" ", "_")
    frame.to_csv(CLEAN_PATH / f"{output_stem}.csv", index=False)

features_path = CLEAN_PATH / "customer_viewing_features.csv"
customer_viewing_features.to_csv(features_path, index=False)

assert customers_clean["customer_id"].is_unique
assert tickets_clean["ticket_date"].notna().all()
assert payments_clean["subscription_id"].notna().all()

cleaning_summary = pd.DataFrame(
    [
        {"check": "datasets exported", "result": len(cleaned_datasets)},
        {"check": "customer duplicate IDs removed", "result": customer_duplicate_rows_removed},
        {"check": "invalid ticket dates removed", "result": invalid_ticket_date_count},
        {"check": "payment IDs reconciled", "result": matched_payment_count},
        {"check": "unmatched payment subscription IDs", "result": unmatched_blank_subscription_count},
        {"check": "customer feature rows", "result": len(customer_viewing_features)},
    ]
)
print(cleaning_summary.to_string(index=False))
print(f"\nCleaned datasets saved in: {CLEAN_PATH}")
print(f"Customer viewing features saved in: {features_path}")

                             check  result
                 datasets exported       9
    customer duplicate IDs removed      48
      invalid ticket dates removed      61
            payment IDs reconciled  110735
unmatched payment subscription IDs       0
             customer feature rows    7980

Cleaned datasets saved in: C:\Users\shukl\Documents\DS project\DataScienceProject\data\processed
Customer viewing features saved in: C:\Users\shukl\Documents\DS project\DataScienceProject\data\processed\customer_viewing_features.csv


### 9.10 Profile the remaining datasets

The next four datasets were only given shared formatting cleanup earlier. Before choosing dataset-specific rules, inspect their shapes, missingness, duplicate rows, data types, and important category values. This prevents us from applying unsuitable imputations to labels or reference data.

In [51]:
remaining_dataset_keys = {
    "churn_labels": dataset_name("churn_labels"),
    "content": dataset_name("content"),
    "customer_feedback": dataset_name("customer_feedback"),
    "subscription_plans": dataset_name("subscription_plans"),
}

remaining_profile = []
for dataset_label in remaining_dataset_keys.values():
    frame = cleaned_datasets[dataset_label]
    remaining_profile.append(
        {
            "dataset": dataset_label,
            "rows": len(frame),
            "columns": len(frame.columns),
            "missing_cells": int(frame.isna().sum().sum()),
            "duplicate_rows": int(frame.duplicated().sum()),
        }
    )

print(pd.DataFrame(remaining_profile).to_string(index=False))

category_columns = {
    "churn_labels": ["churned", "churn_reason"],
    "content": ["content_type", "genre", "content_language", "production_country", "maturity_rating"],
    "customer_feedback": ["rating", "sentiment_label"],
    "subscription_plans": ["plan_name", "video_quality", "advertisements"],
}

for dataset_type, columns in category_columns.items():
    dataset_label = remaining_dataset_keys[dataset_type]
    frame = cleaned_datasets[dataset_label]
    print(f"\n{dataset_label} category values:")
    for column in columns:
        if column in frame.columns:
            print(f"\n{column}:")
            print(frame[column].value_counts(dropna=False).head(20).to_string())

              dataset  rows  columns  missing_cells  duplicate_rows
         churn_labels  8000        4          12018               0
              content   500        9              0               0
customer_feedback (2)  5046        6              0               0
   subscription_plans     3        6              0               0

churn_labels category values:

churned:
churned
False    6009
True     1991

churn_reason:
churn_reason
NaN                                 6009
Too expensive                        365
Found alternative platform           284
Not enough content                   217
No longer needed                     212
Technical issues                     182
Household budget cuts                145
Content quality decline              139
Payment issues                       139
Poor customer service experience     127
Moving / relocation                   98
Password sharing crackdown            83

content category values:

content_type:
content_type
Movie    

### 9.11 Clean churn labels

`churned` is the target label, so normalize recognized true/false values and remove rows whose target is invalid or missing. Parse `churn_date`; a missing date for a non-churned customer is expected, not a data defect. Set `churn_reason` to `Not Applicable` for non-churns and `Unknown` only for churned rows whose reason is missing. Keep one row per `customer_id`, preferring the most complete record.

**Interview reasoning:** Target labels must not be median-imputed. Distinguishing “not applicable” from “unknown” preserves the meaning of missingness.

In [52]:
remaining_cleaning_report = []


def deduplicate_by_key(frame, key_column, date_column=None):
    rows_before = len(frame)
    sort_columns = ["_completeness"]
    ascending = [False]
    if date_column:
        sort_columns.append(date_column)
        ascending.append(False)

    result = frame.copy()
    result["_completeness"] = result.notna().sum(axis=1)
    result = (
        result.sort_values(
            sort_columns,
            ascending=ascending,
            na_position="last",
            kind="mergesort",
        )
        .dropna(subset=[key_column])
        .drop_duplicates(subset=key_column, keep="first")
        .drop(columns="_completeness")
        .reset_index(drop=True)
    )
    return result, rows_before - len(result)


churn_key = remaining_dataset_keys["churn_labels"]
churn_clean = cleaned_datasets[churn_key].copy()
churn_rows_before = len(churn_clean)
target_values = churn_clean["churned"].astype("string").str.strip().str.lower()
churn_clean["churned"] = target_values.map(
    {
        "true": True,
        "1": True,
        "yes": True,
        "false": False,
        "0": False,
        "no": False,
    }
).astype("boolean")
invalid_churn_target_count = int(churn_clean["churned"].isna().sum())
churn_clean = churn_clean.dropna(subset=["churned"]).copy()
churn_clean["churned"] = churn_clean["churned"].astype(bool)

raw_churn_dates = churn_clean["churn_date"]
churn_clean["churn_date"] = pd.to_datetime(
    raw_churn_dates, errors="coerce"
)
invalid_churn_date_count = int(
    (raw_churn_dates.notna() & churn_clean["churn_date"].isna()).sum()
)
non_churn_date_count = int(
    ((~churn_clean["churned"]) & churn_clean["churn_date"].notna()).sum()
)
churn_clean.loc[~churn_clean["churned"], "churn_date"] = pd.NaT

churn_clean.loc[~churn_clean["churned"], "churn_reason"] = "Not Applicable"
churn_reason_missing = churn_clean["churn_reason"].isna()
churn_clean.loc[
    churn_clean["churned"] & churn_reason_missing, "churn_reason"
] = "Unknown"

churn_clean, churn_rows_removed = deduplicate_by_key(
    churn_clean, "customer_id", "churn_date"
)
cleaned_datasets[churn_key] = churn_clean
remaining_cleaning_report.append(
    {
        "dataset": churn_key,
        "rows_before": churn_rows_before,
        "rows_after": len(churn_clean),
        "invalid_targets": invalid_churn_target_count,
        "invalid_dates": invalid_churn_date_count,
        "non_churn_dates_cleared": non_churn_date_count,
        "rows_removed": churn_rows_removed,
    }
)

print(f"Churn rows before cleaning: {churn_rows_before}")
print(f"Invalid/missing churn targets removed: {invalid_churn_target_count}")
print(f"Malformed churn dates set missing: {invalid_churn_date_count}")
print(f"Unexpected non-churn dates cleared: {non_churn_date_count}")
print(f"Rows removed for duplicate/missing customer IDs: {churn_rows_removed}")
print(churn_clean["churned"].value_counts(dropna=False).to_string())

Churn rows before cleaning: 8000
Invalid/missing churn targets removed: 0
Malformed churn dates set missing: 0
Unexpected non-churn dates cleared: 0
Rows removed for duplicate/missing customer IDs: 0
churned
False    6009
True     1991


### 9.12 Clean content metadata

Normalize text categories to consistent casing and map known country aliases to canonical names. Convert `release_year` and `duration_minutes` to numeric values; malformed or non-positive values become missing rather than being guessed. Keep one row per `content_id`, preferring the most complete record.

**Strategy note:** Do not cap content duration just because it is large. A long film or series entry can be valid; here we validate the domain (positive duration) instead of treating every statistical extreme as an error.

In [54]:
content_key = remaining_dataset_keys["content"]
content_clean = cleaned_datasets[content_key].copy()
content_rows_before = len(content_clean)
content_type_aliases = {"tv show": "TV Show"}

content_clean["content_type"] = content_clean["content_type"].map(
    lambda value: normalize_category(value, content_type_aliases)
)
for column in ["genre", "content_language"]:
    content_clean[column] = content_clean[column].map(
        lambda value: normalize_category(value, {})
    )
content_clean["production_country"] = content_clean["production_country"].map(
    lambda value: normalize_category(value, country_aliases)
)
content_clean["maturity_rating"] = content_clean["maturity_rating"].map(
    lambda value: value.strip().upper() if isinstance(value, str) else value
)

content_value_report = []
for column in ["release_year", "duration_minutes"]:
    original_values = content_clean[column]
    numeric_values = pd.to_numeric(original_values, errors="coerce")
    malformed_count = int(
        (original_values.notna() & numeric_values.isna()).sum()
    )
    nonpositive_count = int(
        (numeric_values.notna() & (numeric_values <= 0)).sum()
    )
    content_clean[column] = numeric_values.mask(numeric_values <= 0)
    content_value_report.append(
        {
            "column": column,
            "malformed_values": malformed_count,
            "non_positive_values": nonpositive_count,
        }
    )

content_clean, content_rows_removed = deduplicate_by_key(
    content_clean, "content_id"
)
cleaned_datasets[content_key] = content_clean
remaining_cleaning_report.append(
    {
        "dataset": content_key,
        "rows_before": content_rows_before,
        "rows_after": len(content_clean),
        "rows_removed": content_rows_removed,
    }
)

print(pd.DataFrame(content_value_report).to_string(index=False))
print(f"Content rows removed for duplicate/missing IDs: {content_rows_removed}")
print("Standardized content types:")
print(content_clean["content_type"].value_counts(dropna=False).to_string())

          column  malformed_values  non_positive_values
    release_year                 0                    0
duration_minutes                 0                    0
Content rows removed for duplicate/missing IDs: 0
Standardized content types:
content_type
Movie               216
TV Show             144
Limited Series       59
Documentary          55
Stand-Up Special     26


### 9.13 Clean customer feedback

Parse `feedback_date`, standardize sentiment labels, and ensure ratings are numeric integers from 1 to 5. Invalid ratings become missing; do not impute a customer's opinion. Keep feedback with an invalid date but set that date missing, since the rating and text can still be useful. Deduplicate by `feedback_id`, preferring the most complete record and then the latest valid date.

In [55]:
feedback_key = remaining_dataset_keys["customer_feedback"]
feedback_clean = cleaned_datasets[feedback_key].copy()
feedback_rows_before = len(feedback_clean)

raw_feedback_dates = feedback_clean["feedback_date"]
feedback_clean["feedback_date"] = pd.to_datetime(
    raw_feedback_dates, errors="coerce"
)
invalid_feedback_date_count = int(
    (raw_feedback_dates.notna() & feedback_clean["feedback_date"].isna()).sum()
)

raw_ratings = feedback_clean["rating"]
numeric_ratings = pd.to_numeric(raw_ratings, errors="coerce")
invalid_rating_mask = (
    numeric_ratings.isna()
    | (numeric_ratings < 1)
    | (numeric_ratings > 5)
    | (numeric_ratings % 1 != 0)
)
invalid_rating_count = int(invalid_rating_mask.sum())
feedback_clean["rating"] = numeric_ratings.mask(invalid_rating_mask).astype("Int64")
feedback_clean["sentiment_label"] = feedback_clean["sentiment_label"].map(
    lambda value: normalize_category(value, {})
)

feedback_clean, feedback_rows_removed = deduplicate_by_key(
    feedback_clean, "feedback_id", "feedback_date"
)
cleaned_datasets[feedback_key] = feedback_clean
remaining_cleaning_report.append(
    {
        "dataset": feedback_key,
        "rows_before": feedback_rows_before,
        "rows_after": len(feedback_clean),
        "invalid_ratings": invalid_rating_count,
        "invalid_dates": invalid_feedback_date_count,
        "rows_removed": feedback_rows_removed,
    }
)

print(f"Invalid/out-of-range ratings set missing: {invalid_rating_count}")
print(f"Malformed feedback dates set missing: {invalid_feedback_date_count}")
print(f"Feedback rows removed for duplicate/missing IDs: {feedback_rows_removed}")
print(feedback_clean["rating"].value_counts(dropna=False).sort_index().to_string())

Invalid/out-of-range ratings set missing: 0
Malformed feedback dates set missing: 0
Feedback rows removed for duplicate/missing IDs: 0
rating
1     725
2     711
3    1599
4    1032
5     979


### 9.14 Clean subscription plan reference data

Standardize plan names, video-quality labels, and advertisement values. Convert `monthly_price` to a nonnegative number and require `max_devices` to be a positive integer. Invalid values become missing and are reported; do not impute a tiny three-row reference table. Keep one row per `plan_id`.

**Interview reasoning:** Reference tables should have stable keys and valid domains. With only three plans, statistical imputation would be misleading; missing plan attributes should be checked against the source of truth.

In [56]:
plans_key = remaining_dataset_keys["subscription_plans"]
plans_clean = cleaned_datasets[plans_key].copy()
plans_rows_before = len(plans_clean)

plans_clean["plan_name"] = plans_clean["plan_name"].map(
    lambda value: normalize_category(value, {})
)
quality_aliases = {
    "720p hd": "720p (HD)",
    "1080p full hd": "1080p (Full HD)",
    "4k ultra hd": "4K (Ultra HD)",
}
plans_clean["video_quality"] = plans_clean["video_quality"].map(
    lambda value: normalize_category(value, quality_aliases)
)
advertisement_aliases = {
    "yes": "Yes",
    "true": "Yes",
    "no": "No",
    "false": "No",
}
plans_clean["advertisements"] = plans_clean["advertisements"].map(
    lambda value: normalize_category(value, advertisement_aliases)
)

plan_value_report = []
for column, allow_zero in [("monthly_price", True), ("max_devices", False)]:
    original_values = plans_clean[column]
    numeric_values = pd.to_numeric(original_values, errors="coerce")
    invalid_mask = numeric_values.isna()
    if allow_zero:
        invalid_mask |= numeric_values < 0
    else:
        invalid_mask |= (numeric_values <= 0) | (numeric_values % 1 != 0)
    invalid_count = int(invalid_mask.sum())
    plans_clean[column] = numeric_values.mask(invalid_mask)
    if column == "max_devices":
        plans_clean[column] = plans_clean[column].astype("Int64")
    plan_value_report.append(
        {"column": column, "invalid_values_set_missing": invalid_count}
    )

plans_clean, plan_rows_removed = deduplicate_by_key(plans_clean, "plan_id")
cleaned_datasets[plans_key] = plans_clean
remaining_cleaning_report.append(
    {
        "dataset": plans_key,
        "rows_before": plans_rows_before,
        "rows_after": len(plans_clean),
        "rows_removed": plan_rows_removed,
    }
)

print(pd.DataFrame(plan_value_report).to_string(index=False))
print(f"Plan rows removed for duplicate/missing IDs: {plan_rows_removed}")
print("Clean subscription plans:")
print(plans_clean.to_string(index=False))

       column  invalid_values_set_missing
monthly_price                           0
  max_devices                           0
Plan rows removed for duplicate/missing IDs: 0
Clean subscription plans:
plan_id plan_name  monthly_price   video_quality  max_devices advertisements
 PLAN01     Basic           6.99       720p (HD)            1            Yes
 PLAN02  Standard          15.49 1080p (Full HD)            2             No
 PLAN03   Premium          22.99   4K (Ultra HD)            4             No


### 9.15 Export and verify all cleaned datasets

Save all nine datasets after their applicable cleaning steps. This final export overwrites the checkpoint from section 9.9. Assertions check key uniqueness and key value domains, while the report summarizes what changed in the four additional datasets.

In [58]:
assert churn_clean["customer_id"].is_unique
assert churn_clean["churned"].notna().all()
assert content_clean["content_id"].is_unique
assert feedback_clean["feedback_id"].is_unique
assert plans_clean["plan_id"].is_unique
assert plans_clean["monthly_price"].dropna().ge(0).all()
assert plans_clean["max_devices"].dropna().gt(0).all()

for dataset_label, frame in cleaned_datasets.items():
    output_stem = re.sub(r"\s*\(\d+\)$", "", dataset_label).replace(" ", "_")
    frame.to_csv(CLEAN_PATH / f"{output_stem}.csv", index=False)

customer_viewing_features.to_csv(
    CLEAN_PATH / "customer_viewing_features.csv", index=False
)

remaining_cleaning_summary = (
    pd.DataFrame(remaining_cleaning_report)
    .drop_duplicates(subset="dataset", keep="last")
    .reset_index(drop=True)
)
print("Additional dataset cleaning summary:")
print(remaining_cleaning_summary.fillna("").to_string(index=False))
print(f"\nCleaned datasets exported: {len(cleaned_datasets)}")
print(f"Output directory: {CLEAN_PATH}")

Additional dataset cleaning summary:
              dataset  rows_before  rows_after invalid_targets invalid_dates non_churn_dates_cleared  rows_removed invalid_ratings
         churn_labels         8000        8000             0.0           0.0                     0.0             0                
              content          500         500                                                                   0                
customer_feedback (2)         5046        5046                           0.0                                     0             0.0
   subscription_plans            3           3                                                                   0                

Cleaned datasets exported: 9
Output directory: C:\Users\shukl\Documents\DS project\DataScienceProject\data\processed
